# Pendulum Examples

`pendulum` is a drop-in replacement for Python's `datetime` that makes timezone-aware datetimes easy — which is why Airflow uses it for `start_date` in DAGs (see `hello_world_gcs.py`).

Run: `pip install pendulum` if it's not already installed.

In [1]:
import pendulum

## 1. Creating a timezone-aware datetime

This is exactly what happens on line 17 of `hello_world_gcs.py`:
```python
start_date=pendulum.datetime(2024, 1, 1, tz="UTC")
```

In [2]:
dt = pendulum.datetime(2024, 1, 1, tz="UTC")
print(dt)
print(type(dt))
print("Has timezone info:", dt.tzinfo is not None)

2024-01-01 00:00:00+00:00
<class 'pendulum.datetime.DateTime'>
Has timezone info: True


## 2. Compare with a naive stdlib datetime

A naive datetime has no timezone attached — this is what Airflow rejects (or silently mishandles) for `start_date`.

In [3]:
import datetime

naive = datetime.datetime(2024, 1, 1)
print(naive, "-> tzinfo:", naive.tzinfo)

aware = pendulum.datetime(2024, 1, 1, tz="UTC")
print(aware, "-> tzinfo:", aware.tzinfo)

2024-01-01 00:00:00 -> tzinfo: None
2024-01-01 00:00:00+00:00 -> tzinfo: UTC


## 3. Current time in different timezones

In [4]:
utc_now = pendulum.now("UTC")
hk_now = pendulum.now("Asia/Hong_Kong")

print("UTC:", utc_now)
print("Hong Kong:", hk_now)

UTC: 2026-08-29 08:15:29.726008+00:00
Hong Kong: 2026-08-29 16:15:29.726060+08:00


## 4. Converting between timezones

In [5]:
dt_utc = pendulum.datetime(2024, 6, 15, 9, 0, tz="UTC")
dt_hk = dt_utc.in_timezone("Asia/Hong_Kong")

print("UTC:", dt_utc)
print("Hong Kong:", dt_hk)

UTC: 2024-06-15 09:00:00+00:00
Hong Kong: 2024-06-15 17:00:00+08:00


## 5. Date arithmetic (add/subtract)

Useful for things like computing schedule windows or backfill ranges.

In [ ]:
start = pendulum.datetime(2024, 1, 1, tz="UTC")

print("Plus 1 day:", start.add(days=1))
print("Plus 1 month:", start.add(months=1))
print("Minus 1 week:", start.subtract(weeks=1))

## 6. Handling DST correctly

Pendulum accounts for daylight saving time transitions automatically — stdlib `datetime` + naive arithmetic often gets this wrong.

In [6]:
before_dst = pendulum.datetime(2024, 3, 9, 12, tz="America/New_York")
after_dst = before_dst.add(days=2)

print(before_dst, "UTC offset:", before_dst.utcoffset())
print(after_dst, "UTC offset:", after_dst.utcoffset())

2024-03-09 12:00:00-05:00 UTC offset: -1 day, 19:00:00
2024-03-11 12:00:00-04:00 UTC offset: -1 day, 20:00:00


## 7. Parsing strings

In [7]:
parsed = pendulum.parse("2024-01-01T09:00:00+00:00")
print(parsed)
print(parsed.to_iso8601_string())

2024-01-01 09:00:00+00:00
2024-01-01T09:00:00+00:00


## 8. Human-friendly differences

Handy for logging things like "DAG run started 3 hours ago".

In [8]:
past = pendulum.now("UTC").subtract(hours=3)
print(past.diff_for_humans())

3 hours ago


## 9. Why this matters for the DAG

In `hello_world_gcs.py`:
```python
@dag(
    dag_id="hello_world_gcs",
    schedule=None,
    start_date=pendulum.datetime(2024, 1, 1, tz="UTC"),
    catchup=False,
    tags=["gcs", "example"],
)
```
Airflow's scheduler compares `start_date` against the current time (in UTC) to decide when the DAG is eligible to run. Since it needs a timezone-aware value to do that comparison safely, `pendulum.datetime(..., tz="UTC")` is the standard, Airflow-recommended way to construct it.